# 02. Virality Prediction Engine: Heuristic Index & Pre-Publication Forecasting

This notebook covers:
1. **Part A: Engagement Virality Index (EVI)** - Ex-post heuristic index measuring high-intent interaction density per unit exposure.
2. **Part B: Target Leakage Prevention** - Strict separation of pre-publication features from post-publication outcome metrics.
3. **Part C: Chronological Train/Test Splitting** - Preventing lookahead bias.
4. **Part D: Baseline Benchmarking** - Comparing Ridge Regression against a naive Dummy Baseline.
5. **Part E: Pre-Publication Draft Forecasting** - Estimating potential reach and performance tier before publishing.


In [ ]:
import sys
sys.path.append('..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.modeling.virality_predictor import ViralityPredictor, PRE_PUBLICATION_FEATURES, OUTCOME_METRICS
from src.preprocessing.cleaning import get_latest_posts_view

# 1. Load Processed Dataset
df = pd.read_parquet('../data/processed/processed_posts.parquet')
print(f'Total processed records: {len(df)}')
latest_posts = get_latest_posts_view(df)
print(f'Unique post entities: {len(latest_posts)}')


## Part A: Ex-Post Engagement Virality Index (EVI)
We compute the availability-aware EVI across posts, weighting Shares (40%), Saves (35%), Comments (15%), and Likes (10%), with retention adjustments on videos.


In [ ]:
predictor = ViralityPredictor()
scores_df = predictor.calculate_engagement_virality_index_detailed(latest_posts)
latest_posts['engagement_virality_index'] = scores_df['engagement_virality_index']
latest_posts['confidence_tier'] = scores_df['confidence_tier']
latest_posts['denominator_used'] = scores_df['denominator_used']

print('Engagement Virality Index Statistics:')
print(latest_posts['engagement_virality_index'].describe())

topic_rankings = predictor.rank_topics_by_virality(latest_posts)
topic_rankings


## Part B: Pre-Publication Model Training & Baseline Benchmarking
We train a regularized Ridge Regressor strictly on pre-publication features and evaluate on held-out temporal data against a Dummy baseline.


In [ ]:
meta = predictor.train_and_evaluate(latest_posts, model_type='ridge', test_ratio=0.2)

print('=== TEMPORAL EVALUATION RESULTS ===')
print(f"Train samples: {meta['train_samples_count']} | Test samples: {meta['test_samples_count']}")
print(f"Dummy Baseline -> MAE: {meta['baseline_metrics']['mae']}, RMSE: {meta['baseline_metrics']['rmse']}")
print(f"Ridge Model    -> MAE: {meta['candidate_metrics']['mae']}, RMSE: {meta['candidate_metrics']['rmse']}")
print(f"MAE Improvement: {meta['mae_lift_pct']}%")
print(f"RMSE Improvement: {meta['rmse_lift_pct']}%")
print(f"Spearman Rank Correlation: {meta['candidate_metrics']['spearman_rho']}")
print(f"Beats Baseline: {meta['beats_baseline']}")


## Part C: Feature Importance (Standardized Coefficients)
Analyzing which pre-publication variables drive the largest expected change in Engagement Virality Index.


In [ ]:
top_feats = pd.Series(meta['top_features']).sort_values(ascending=True)
plt.figure(figsize=(9, 5))
top_feats.plot(kind='barh', color='steelblue')
plt.title('Top Pre-Publication Coefficients (Ridge Regression)')
plt.xlabel('Standardized Coefficient')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()


## Part D: Pre-Publication Draft Forecasting Simulator
Simulate expected performance for an unreleased concept draft.


In [ ]:
draft = pd.DataFrame([{
    'platform': 'instagram',
    'content_type': 'reel',
    'topic': 'Social Anxiety',
    'hook_type': 'Visual Hook',
    'duration_seconds': 30,
    'posting_hour': 18,
    'posting_day_of_week': 'Sunday',
    'follower_count': 45000
}])

prediction_res = predictor.predict_pre_publication(draft)
prediction_res
